In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
from Database.TPData import TPData
from Math.ti_class import TI_class, VI_class, TR_class
from Math.lm_class import kalman, LinearModel
from Math.accumfeatures import MA, MSTD, DifferentialEMA, DerivativeEMA
from Strategies.Market_making.model_class import simple_model
tol=(1e-1)/2

from Strategies.Strategy_base import Strategy_base
from Strategies.Backtester_class import BacktesterClass
from Strategies.Strategy_MeanR_NT import Strategy_MeanR_NT

In [2]:
import pickle
with open("de_m1q1_23.pkl", 'rb') as pickle_file:
    data_p = pickle.load(pickle_file)

In [ ]:
history_window = 20
tp, sl = 1, 1
trend_c = .5
bands = 1
tau = 30
tau_ema = 30
tau_sigma = 15
mstd_n = 30

In [10]:
from Strategies.Models.model_trend import Model_trend, Model_mstd

back_test = BacktesterClass('m', trade_type='mm')
model_t = Model_trend(15, 30)
model_m = Model_mstd(15, 30)
models = [model_m, model_t]
strat = Strategy_MeanR_NT([1, 1, .5, 1, 20, 30], models)
back_test.simulate_strategy(data_p, strat, models)

In [11]:
pos = strat.position_storage.export_dict()

In [12]:
pos_df = pd.DataFrame([{'pnl': v['pnl'], 'index': v['open']['index'], 'take_profit': v['take_profit'], 'stop_loss': v['stop_loss']} for k, v in pos.items()])
pos_df.set_index('index', inplace=True)

In [13]:
strat_data = pd.DataFrame(strat.history_stack)
strat_data

,Date,Open,Close,Low,High,count,index,ema,sigma,trend,band_upper,band_lower
0,2023-07-10 11:42:20,-36.380,-36.610000,-36.9300,-36.1500,15,0,NaN,NaN,NaN,NaN,NaN
1,2023-07-10 11:53:19,-36.595,-35.980000,-36.7500,-35.4600,19,1,-21.702456,3.301483,-1.405706,-18.400972,-25.003939
2,2023-07-10 12:19:59,-36.250,-35.845000,-36.4140,-35.6300,20,2,-22.214801,4.117675,0.747888,-18.097126,-26.332475
3,2023-07-10 12:44:15,-35.870,-35.603333,-36.0100,-34.6000,18,3,-22.716792,4.722289,0.039329,-17.994502,-27.439081
4,2023-07-10 13:22:26,-35.600,-35.158000,-36.0500,-34.8600,20,4,-23.199569,5.175616,-0.217281,-18.023953,-28.375185
...,...,...,...,...,...,...,...,...,...,...,...,...
373,2023-08-28 12:28:30,-17.714,-15.850000,-17.7140,-15.8500,12,373,-23.939199,2.964582,-0.000023,-20.974617,-26.903780
374,2023-08-28 14:16:52,-15.640,-19.102500,-19.1025,-14.7575,19,374,-23.763050,2.935802,-0.000026,-20.827247,-26.698852
375,2023-08-28 14:35:53,-19.290,-17.830000,-19.2900,-17.8300,20,375,-23.461921,3.133594,0.000053,-20.328327,-26.595515
376,2023-08-28 15:34:49,-18.700,-17.840000,-18.9200,-16.3400,19,376,-23.190323,3.266132,0.000006,-19.924190,-26.456455


In [14]:
df = pd.concat([strat_data, pos_df], axis=1)
df

,Date,Open,Close,Low,High,count,index,ema,sigma,trend,band_upper,band_lower,pnl,take_profit,stop_loss
0,2023-07-10 11:42:20,-36.380,-36.610000,-36.9300,-36.1500,15,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-07-10 11:53:19,-36.595,-35.980000,-36.7500,-35.4600,19,1,-21.702456,3.301483,-1.405706,-18.400972,-25.003939,NaN,NaN,NaN
2,2023-07-10 12:19:59,-36.250,-35.845000,-36.4140,-35.6300,20,2,-22.214801,4.117675,0.747888,-18.097126,-26.332475,NaN,NaN,NaN
3,2023-07-10 12:44:15,-35.870,-35.603333,-36.0100,-34.6000,18,3,-22.716792,4.722289,0.039329,-17.994502,-27.439081,NaN,NaN,NaN
4,2023-07-10 13:22:26,-35.600,-35.158000,-36.0500,-34.8600,20,4,-23.199569,5.175616,-0.217281,-18.023953,-28.375185,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2023-08-28 12:28:30,-17.714,-15.850000,-17.7140,-15.8500,12,373,-23.939199,2.964582,-0.000023,-20.974617,-26.903780,NaN,NaN,NaN
374,2023-08-28 14:16:52,-15.640,-19.102500,-19.1025,-14.7575,19,374,-23.763050,2.935802,-0.000026,-20.827247,-26.698852,NaN,NaN,NaN
375,2023-08-28 14:35:53,-19.290,-17.830000,-19.2900,-17.8300,20,375,-23.461921,3.133594,0.000053,-20.328327,-26.595515,-2.997247,-23.763050,-17.891445
376,2023-08-28 15:34:49,-18.700,-17.840000,-18.9200,-16.3400,19,376,-23.190323,3.266132,0.000006,-19.924190,-26.456455,NaN,NaN,NaN


In [8]:
from Utilities.plotUtils import bokehPlot
df['returns'] = df['pnl'].fillna(0).cumsum()
bokehPlot(df, title="strategy sl/tp fixed by position",
           col_list=[['Close', 'ema', 'take_profit', 'stop_loss', 'band_upper', 'band_lower'], ['trend']],
           scatter=['take_profit', 'stop_loss'], sub=2)

In [ ]:
df['trend'].plot()

In [ ]:
import pickle
with open('de_m1q1_11.pkl', 'wb') as f:
    pickle.dump(data_p, f)

In [ ]:
n_s = 3
start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 11, 28)
dates = pd.date_range(start_date, end_date, freq='B')
market = ['de', 'de']
tenor = ['m', 'q']
tn1_list = [1, 1]
tn2_list = []
brk_list = ['eex']
mm_bool = [True, False]

start_time = time(9, 0, 0, 0)
end_time = time(17, 25, 0, 0)
gran = None
coeff_list = norm_coeff([1, -1], market)


ob_data = False

df_ba = pd.DataFrame([])
df_tr = pd.DataFrame([])

spread_class = SpreadSingle(market, tenor, tn1_list, tn2_list, brk_list)
data_class = SpreadViewerData()
db_class = TPData()
tenors_list = spread_class.tenors_list
if not ob_data:
    data_class.load_best_order_otc(market, tenors_list,
                                   spread_class.product_dates(dates, n_s),
                                   db_class,
                                   start_time=start_time, end_time=end_time)
else:
    data_class.load_best_ob(market, tenors_list, dates, spread_class.product_dates(dates, n_s),
                            v_thres=5, freq=gran)

data_class_tr = SpreadViewerData()
data_class_tr.load_trades_otc(market, tenors_list, db_class,
                              start_time=start_time, end_time=end_time)


data_dict = spread_class.aggregate_data(data_class, dates, n_s, gran=gran,
                                        start_time=start_time, end_time=end_time)
df_ba = spread_class.spread_maker(data_dict, coeff_list, trade_type=['cmb', 'cmb'])
col_list=['bid', 'ask', 'volume']
trade_dict = spread_class.aggregate_data(data_class_tr, dates, n_s, gran='1S',
                                         start_time=start_time, end_time=end_time,
                                         col_list=col_list, data_dict=data_dict)
df_tr = spread_class.get_trades_otc(data_dict, trade_dict, coeff_list, mm_bool)

data_p = df_tr['price']